# Scraper de datos musicales
Scraping de letras (AZLyrics) y géneros (Last.fm) a partir de un CSV propio.


In [5]:
# Ejecuta esta celda solo la primera vez
%pip install cloudscraper beautifulsoup4 pandas requests

## Imports y configuración

In [1]:
import re
import time
import random
import logging
import pandas as pd
import cloudscraper
import requests
from bs4 import BeautifulSoup, Comment
from urllib.parse import quote

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# Cloudscraper (bypassea Cloudflare en AZLyrics)
SCRAPER = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)

# Cabeceras Last.fm 
HEADERS_LASTFM = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

# Delays entre requests
AZLYRICS_DELAY = (5, 10)   # AZLyrics tiene rate-limiting agresivo
LASTFM_DELAY   = (1, 3)

## Funciones de scraping

In [7]:
# "The Beatles" -> "thebeatles" para usarlo en la URL
def _slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())

def _get_lastfm(url: str):
    time.sleep(random.uniform(*LASTFM_DELAY))
    try:
        resp = SCRAPER.get(url, headers=HEADERS_LASTFM, timeout=15)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"Last.fm request failed: {url} → {e}")
        return None
    
def _get_azlyrics(url: str):
    time.sleep(random.uniform(*AZLYRICS_DELAY))
    try:
        resp = SCRAPER.get(url, timeout=20)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"AZLyrics request failed: {url} → {e}")
        return None

def scrape_lyrics(artist: str, song: str):
    url = f"https://www.azlyrics.com/lyrics/{_slug(artist)}/{_slug(song)}.html"
    log.info(f"AZLyrics → {url}")
    resp = _get_azlyrics(url)
    if resp is None:
        return None
    if "ringtone" not in resp.text:
        log.warning(f"Página sin div.ringtone (canción no indexada)")
        return None
    soup = BeautifulSoup(resp.text, "html.parser")
    # Método 1: hermano de div.ringtone
    ringtone = soup.find("div", class_="ringtone")
    if ringtone:
        div = ringtone.find_next_sibling("div")
        if div:
            text = div.get_text(separator="\n").strip()
            if len(text) > 80:
                log.info(f"Letra encontrada ({len(text)} chars)")
                return text
    # Método 2: comentario HTML
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment) and "Usage of azlyrics" in t):
        div = comment.find_next("div")
        if div:
            text = div.get_text(separator="\n").strip()
            if len(text) > 80:
                return text
    # Método 3: div sin class largo
    for div in soup.find_all("div", class_=False, id=False):
        text = div.get_text(separator="\n").strip()
        if len(text) > 150 and text.count("\n") > 5:
            if not any(s in text[:60] for s in ["AZLyrics", "Submit", "Login", "©"]):
                return text
    log.warning(f"No se encontró letra para {artist} - {song}")
    return None

def _extract_lastfm_tags(html: str, top_n: int):
    soup = BeautifulSoup(html, "html.parser")
    tags = []
    for selector in [".catalogue-tags .tag", ".catalogue-tags a",
                     ".tag-list .tag", ".tags-list a", "ul.tags li a", "a.tag"]:
        for el in soup.select(selector):
            tag = el.get_text(strip=True).lower()
            if tag and tag not in tags and len(tag) > 1:
                tags.append(tag)
        if tags:
            break
    if not tags:
        seen = set()
        for a in soup.find_all("a", href=True):
            if "/tag/" in a["href"]:
                tag = a.get_text(strip=True).lower()
                if tag and tag not in seen and len(tag) > 1:
                    tags.append(tag); seen.add(tag)
    return tags[:top_n]

def scrape_genre(artist: str, song: str = None, top_n: int = 3):
    tags = []
    if song:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}/_/{quote(song)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    if not tags:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    log.info(f"Géneros: {tags}")
    return tags

def scrape_dataset(songs: list, output_csv: str = "music_dataset.csv") -> pd.DataFrame:
    results = []
    for i, entry in enumerate(songs, 1):
        artist, song = entry["artist"], entry["song"]
        log.info(f"[{i}/{len(songs)}] {artist} — {song}")
        lyrics = scrape_lyrics(artist, song)
        genres = scrape_genre(artist, song, top_n=3)
        results.append({
            "artist":  artist, "song": song, "lyrics": lyrics,
            "genres":  ", ".join(genres) if genres else None,
            "genre_1": genres[0] if len(genres) > 0 else None,
            "genre_2": genres[1] if len(genres) > 1 else None,
            "genre_3": genres[2] if len(genres) > 2 else None,
        })
        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8")
    log.info(f"Dataset guardado: {output_csv} ({len(results)} canciones)")
    return pd.DataFrame(results)

print("Funciones cargadas")


Funciones cargadas


## Cargar y unir CSVs

In [8]:
import json
from pathlib import Path

# ── Credenciales Kaggle ──────────────────────────────────────────────────────
KAGGLE_USERNAME = ""   # tu usuario de Kaggle
KAGGLE_KEY      = ""   # tu API key de Kaggle

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
(kaggle_dir / "kaggle.json").chmod(0o600)

# ── Descarga del dataset ─────────────────────────────────────────────────────
%pip install kaggle -q  # descomenta si no tienes kaggle instalado
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi(); api.authenticate()
api.dataset_download_files("neisse/scrapped-lyrics-from-6-genres", path=".", unzip=True, quiet=False)

songs_df   = pd.read_csv("lyrics-data.csv")
artists_df = pd.read_csv("artists-data.csv")

print("lyrics-data.csv columnas:", songs_df.columns.tolist())
print("artists-data.csv columnas:", artists_df.columns.tolist())

# Join por ALink (songs) <-> Link (artists)
merged = songs_df.merge(artists_df[["Link", "Artist"]], left_on="ALink", right_on="Link", how="left")

# Filtrar inglés y construir lista
all_songs = [
    {"artist": row["Artist"].strip(), "song": row["SName"].strip()}
    for _, row in merged.iterrows()
    if pd.notna(row["SName"]) and pd.notna(row["Artist"])
    and row.get("language") == "en"
]

# Rango de canciones a scrapear (índices basados en 1, ambos incluidos)
# Cambia START y END según el tramo que quieras procesar
START = 20000       # primera canción a scrapear
END   = 30000   # última canción a scrapear (None = hasta el final)

songs = all_songs[START - 1 : END][::100]

print(f"Total canciones en inglés: {len(all_songs)}")
print(f"Rango seleccionado:        {START} → {END or len(all_songs)}")
print(f"Canciones a scrapear:      {len(songs)} (cada 100)")
songs[:3]  # preview

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/neisse/scrapped-lyrics-from-6-genres


KeyboardInterrupt: 

## Ejecutar scraping
Con 1000 canciones este proceso tarda **3-5 horas**. El CSV se guarda tras cada canción,
así que puedes interrumpirlo y usar la celda de *Reanudar* para continuar.


In [ ]:
OUTPUT_CSV = f"music_dataset_{START}_{END}.csv"

df = scrape_dataset(songs, output_csv=OUTPUT_CSV)

print(f"Total canciones: {len(df)}")
print(f"Con letra:       {df['lyrics'].notna().sum()}")
print(f"Con géneros:     {df['genres'].notna().sum()}")


2026-05-02 21:15:51,131 [INFO] [1/101] Tarja Turunen — Until Silence
2026-05-02 21:15:51,133 [INFO] AZLyrics → https://www.azlyrics.com/lyrics/tarjaturunen/untilsilence.html
2026-05-02 21:15:58,533 [WARNING] AZLyrics request failed: https://www.azlyrics.com/lyrics/tarjaturunen/untilsilence.html → 404 Client Error: Not Found for url: https://www.azlyrics.com/lyrics/tarjaturunen/untilsilence.html
2026-05-02 21:16:01,874 [WARNING] Last.fm request failed: https://www.last.fm/music/Tarja%20Turunen/_/Until%20Silence → 502 Server Error: Bad Gateway for url: https://www.last.fm/music/Tarja%20Turunen/_/Until%20Silence
2026-05-02 21:16:04,847 [INFO] Géneros: ['symphonic metal', 'female vocalists', 'female fronted metal']
2026-05-02 21:16:04,852 [INFO] [2/101] Susan Boyle — You Have To Be There
2026-05-02 21:16:04,853 [INFO] AZLyrics → https://www.azlyrics.com/lyrics/susanboyle/youhavetobethere.html
2026-05-02 21:16:10,515 [INFO] Letra encontrada (1520 chars)
2026-05-02 21:16:13,316 [INFO] Género

KeyboardInterrupt: 

## Reanudar scraping interrumpido

In [ ]:
import os

OUTPUT_CSV = "music_dataset.csv"

if os.path.exists(OUTPUT_CSV):
    done_df   = pd.read_csv(OUTPUT_CSV)
    done_keys = set(zip(done_df["artist"].str.lower(), done_df["song"].str.lower()))
    pending   = [s for s in songs if (s["artist"].lower(), s["song"].lower()) not in done_keys]
    print(f"Ya procesadas: {len(done_df)} | Pendientes: {len(pending)}")

    if pending:
        new_df   = scrape_dataset(pending, output_csv="music_dataset_new.csv")
        combined = pd.concat([done_df, new_df], ignore_index=True)
        combined.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
        print(f"Dataset combinado: {len(combined)} canciones")
    else:
        print("Todas las canciones ya están procesadas")
else:
    print("No existe music_dataset.csv, ejecuta primero la celda de scraping")


## Explorar resultados

In [ ]:
df = pd.read_csv("music_dataset.csv")

print(f"Total canciones:  {len(df)}")
print(f"Con letra:        {df['lyrics'].notna().sum()} ({df['lyrics'].notna().mean():.0%})")
print(f"Con géneros:      {df['genres'].notna().sum()} ({df['genres'].notna().mean():.0%})")
print(f"\nGéneros más frecuentes:")
print(df["genre_1"].value_counts().head(10).to_string())


Total canciones:  101
Con letra:        66 (65%)
Con géneros:      71 (70%)

Géneros más frecuentes:
genre_1
country            44
pop                 4
bluegrass           4
rockabilly          2
oldies              2
60s                 2
symphonic metal     1
electropop          1
christmas           1
perquien            1


In [ ]:
# Vista previa de las primeras filas
df[["artist", "song", "genres", "lyrics"]].head(10)

,artist,song,genres,lyrics
0,Tarja Turunen,Until Silence,"symphonic metal, female vocalists, female fron...",NaN
1,Susan Boyle,You Have To Be There,"pop, love it, female vocalists",What is it Lord that you want\n\nThat I am not...
2,Taylor Swift,Cruel Summer,"electropop, pop, synthpop","(Yeah, yeah, yeah, yeah)\n\n\n\nFever dream hi..."
3,Taylor Swift,London Boy,NaN,[Idris Elba and James Corden:]\n\nWe can go dr...
4,Taylor Swift,Sugar,NaN,What a thing to see\n\nWhat a thing to be\n\nW...
5,Shania Twain,Forever And For Always,"country, pop, female vocalists","Mm, mm\n\nMm, in your arms\n\nOh\n\nI can hear..."
6,Shania Twain,Wild and Wicked,NaN,Full moon cravin' just a little\n\nYoung love ...
7,Johnny Cash,A Croft In Clachan,NaN,NaN
8,Johnny Cash,Devil's Right Hand,"country, folk, singer-songwriter",NaN
9,Johnny Cash,I Heard The Bells On Christmas Day,"christmas, country, folk",I heard the bells on Christmas Day\n\nTheir ol...


## Enriquecer dataset con años de publicación (Wikipedia)
Esta función busca el año de publicación de cada canción en Wikipedia.
Lee `music_dataset.csv`, añade la columna `year` y guarda el resultado
en el mismo archivo (o en uno nuevo si se especifica).

In [2]:
import re
import time
import random
import logging
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote

# Configuración básica de logging si no la tienes
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
log = logging.getLogger(__name__)

# ── Cabeceras para Wikipedia ──────────────────────────────────────────────────
HEADERS_WIKI = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
WIKI_DELAY = (1, 3)  # Wikipedia tiene límites de uso razonables


def _search_wikipedia_title(artist: str, song: str) -> str | None:
    """
    Usa la API de búsqueda de Wikipedia para encontrar el título
    del artículo más relevante para una canción dada.
    """
    query = f"{song} {artist} song"
    api_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": 5,
        "format": "json",
    }
    try:
        time.sleep(random.uniform(*WIKI_DELAY))
        resp = requests.get(api_url, params=params, headers=HEADERS_WIKI, timeout=10)
        resp.raise_for_status()
        results = resp.json().get("query", {}).get("search", [])
        if not results:
            return None
            
        song_slug = song.lower()
        artist_slug = artist.lower()
        
        # 1. Descartar la página principal del artista
        valid_results = [r for r in results if r["title"].lower() != artist_slug]
        if not valid_results:
            return None

        # 2. Priorizar resultados donde el título tenga la canción Y el snippet confirme que es una canción
        music_keywords = ["song", "single", "track"]
        for r in valid_results:
            title_lower = r["title"].lower()
            snippet_lower = r.get("snippet", "").lower()
            
            if song_slug in title_lower and any(kw in snippet_lower for kw in music_keywords):
                return r["title"]

        # 3. Fallback: El primer resultado válido que contenga el título de la canción
        for r in valid_results:
            if song_slug in r["title"].lower():
                return r["title"]
                
        # 4. Último recurso: El primer resultado que no sea el artista
        return valid_results[0]["title"]
        
    except Exception as e:
        log.warning(f"Wikipedia search error ({artist} - {song}): {e}")
        return None


def _extract_year_from_wiki(html: str) -> int | None:
    """
    Extrae el año de publicación de una página Wikipedia.
    Intenta varios métodos en orden de fiabilidad.
    """
    soup = BeautifulSoup(html, "html.parser")

    # ── Método 1: infobox → fila 'Released' ──────────────────────────────────
    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
    if infobox:
        for row in infobox.find_all("tr"):
            header = row.find("th")
            if header and "released" in header.get_text(strip=True).lower():
                td = row.find("td")
                if td:
                    text = td.get_text(" ", strip=True)
                    m = re.search(r"\b(19[5-9]\d|20[0-2]\d)\b", text)
                    if m:
                        return int(m.group(1))

    # ── Método 2: <time> con datetime= en cualquier infobox ──────────────────
    for time_tag in soup.select(".infobox time[datetime]"):
        m = re.search(r"\b(19[5-9]\d|20[0-2]\d)\b", time_tag["datetime"])
        if m:
            return int(m.group(1))

    # ── Método 3: primer párrafo del artículo (Mejorado) ──────────────────────
    keywords = ["released", "single", "recorded", "song", "track", "album"]
    for p in soup.select("#mw-content-text .mw-parser-output > p"):
        text = p.get_text()
        if len(text) < 30:
            continue
            
        # Solo busca el año si el párrafo tiene contexto musical
        if any(kw in text.lower() for kw in keywords):
            m = re.search(r"\b(19[5-9]\d|20[0-2]\d)\b", text)
            if m:
                return int(m.group(1))

    return None


def scrape_year(artist: str, song: str) -> int | None:
    """
    Devuelve el año de publicación de una canción a partir de su
    página de Wikipedia, o None si no se puede determinar.
    """
    title = _search_wikipedia_title(artist, song)
    if not title:
        log.warning(f"Wikipedia: no search result for '{artist} - {song}'")
        return None

    page_url = f"https://en.wikipedia.org/wiki/{quote(title.replace(' ', '_'))}"
    log.info(f"Wikipedia → {page_url}")
    try:
        time.sleep(random.uniform(*WIKI_DELAY))
        resp = requests.get(page_url, headers=HEADERS_WIKI, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        log.warning(f"Wikipedia fetch error: {e}")
        return None

    year = _extract_year_from_wiki(resp.text)
    log.info(f"Año encontrado: {year}" if year else "Año no encontrado")
    return year


def enrich_with_years(
    input_csv: str = "music_dataset.csv",
    output_csv: str | None = None,
    overwrite_existing: bool = False,
) -> pd.DataFrame:
    """
    Lee el dataset completado, añade la columna 'year' buscando
    en Wikipedia la fecha de publicación de cada canción y guarda
    el resultado.
    """
    output_csv = output_csv or input_csv
    
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        log.error(f"El archivo {input_csv} no existe. Por favor, asegúrate de tenerlo en el directorio.")
        return pd.DataFrame()

    if "year" not in df.columns:
        df["year"] = None

    total = len(df)
    pending_mask = df["year"].isna() if not overwrite_existing else pd.Series([True] * total)
    pending_idx  = df.index[pending_mask].tolist()

    log.info(f"Total canciones: {total} | Con año: {total - len(pending_idx)} | Pendientes: {len(pending_idx)}")

    for count, idx in enumerate(pending_idx, 1):
        artist = df.at[idx, "artist"]
        song   = df.at[idx, "song"]
        log.info(f"[{count}/{len(pending_idx)}] {artist} — {song}")

        df.at[idx, "year"] = scrape_year(artist, song)

        # Guardado incremental
        if count % 10 == 0 or count == len(pending_idx):
            df.to_csv(output_csv, index=False, encoding="utf-8")
            log.info(f"Guardado parcial: {output_csv} ({count}/{len(pending_idx)} procesadas)")

    log.info(f"\nEnriquecimiento completado. Canciones con año: {df['year'].notna().sum()}/{total}")
    return df


print("Funciones de enriquecimiento con año cargadas")

Funciones de enriquecimiento con año cargadas


In [3]:
# ── Ejecutar enriquecimiento ──────────────────────────────────────────────────
# Por defecto lee y sobreescribe 'music_dataset.csv'.
# Cambia output_csv si prefieres conservar el original:
#   df = enrich_with_years(output_csv="music_dataset_with_years.csv")

df = enrich_with_years(
        input_csv="music_dataset.csv",
        output_csv="music_dataset.csv",   # sobreescribe el original
        overwrite_existing=False,          # omite filas que ya tienen año
    )

if not df.empty:
    print(f"\nResumen:")
    print(f"  Total canciones:     {len(df)}")
    print(f"  Con año:             {df['year'].notna().sum()} ({df['year'].notna().mean():.0%})")
    print(f"  Sin año:             {df['year'].isna().sum()}")
    print(f"\nDistribución por década:")
    try:
        print(df['year'].dropna().astype(int).floordiv(10).mul(10).value_counts().sort_index().to_string())
    except Exception:
        print("No hay años suficientes para mostrar la distribución.")


2026-05-05 16:34:47,686 [INFO] Total canciones: 99 | Con año: 0 | Pendientes: 99
2026-05-05 16:34:47,686 [INFO] [1/99] Ivete Sangalo — Careless Whisper
2026-05-05 16:34:49,118 [INFO] Wikipedia → https://en.wikipedia.org/wiki/Brian_McKnight_discography
2026-05-05 16:34:52,398 [INFO] Año encontrado: 1992
2026-05-05 16:34:52,399 [INFO] [2/99] Beyoncé — Bootylicious
2026-05-05 16:34:54,075 [INFO] Wikipedia → https://en.wikipedia.org/wiki/Bootylicious
2026-05-05 16:34:55,766 [INFO] Año encontrado: 2001
2026-05-05 16:34:55,767 [INFO] [3/99] Beyoncé — New Shoes
2026-05-05 16:34:59,237 [INFO] Wikipedia → https://en.wikipedia.org/wiki/Party_%28Beyonc%C3%A9_song%29
2026-05-05 16:35:02,457 [INFO] Año encontrado: 2011
2026-05-05 16:35:02,458 [INFO] [4/99] Beyoncé — You Changed
2026-05-05 16:35:04,521 [INFO] Wikipedia → https://en.wikipedia.org/wiki/Dangerously_in_Love
2026-05-05 16:35:07,617 [INFO] Año encontrado: 2003
2026-05-05 16:35:07,619 [INFO] [5/99] Bruno Mars — There I Go Again
2026-05-05 

KeyboardInterrupt: 